# 📘 Prompt Engineering — System Prompts, Personas & Structured Output

A professional intermediate-level lesson on engineering consistent LLM behavior using system prompts, persona design, and structured JSON output via the OpenAI API. Students learn the messages array structure, write precise personas, and extract clean structured data — all in runnable Python code.

---

**How to use this notebook:**

1. Read each section — explanations are in the grey text cells.
2. Run the code cells with **Shift + Enter**.
3. Edit and experiment — the best way to learn is to break things.
4. Complete the **Practice Exercises** at the bottom.

> 💡 This notebook accompanies the teaching video on *Prompt Engineering — System Prompts, Personas & Structured Output*.

## 📑 Table of Contents

1. [The Two Message Roles — System vs User](#the-two-message-roles-—-system-vs-user)
2. [What the LLM Sees on Every Single API Call](#what-the-llm-sees-on-every-single-api-call)
3. [System Prompt and Persona in Action](#system-prompt-and-persona-in-action)
4. [What Makes an Effective Persona](#what-makes-an-effective-persona)
5. [Same Model — Completely Different AI](#same-model-—-completely-different-ai)
6. [The Core Insight](#the-core-insight)
7. [Structured JSON Output](#structured-json-output)
8. [When to Reach for Each Technique](#when-to-reach-for-each-technique)
9. [What You Can Now Engineer](#what-you-can-now-engineer)
10. [Practice Exercises](#practice-exercises)


---

## The Two Message Roles — System vs User

Every LLM API call passes a messages list — an ordered array where each entry has a role and content. The system role is the first message in the list.

### Key Points

- messages list — ordered array, role and content
- system role — permanent instructions, entire conversation
- user role — your actual request, changes every turn
- assistant role — prior replies, conversation history

> 💡 **Analogy:** The system prompt is an employee handbook the AI reads before its first shift. The user prompt is the task a customer brings in. Change the handbook and you have a completely different employee — even when the customer's request is identical.


## System Prompt and Persona in Action

Give the LLM a strict code-reviewer persona via a system prompt and watch how it changes the entire response style

### Step-by-Step Breakdown

**The system message — locking in the persona** — The system message sits at position zero in the messages list. Three instructions are packed into one string: a role (senior Python engineer doing code review), a format rule (bullet points, maximum 3), and a tone constraint (no praise, be direct). Every response the model produces in this session will follow these rules — not because of the user message, but because the system prompt established them as permanent conditions before any conversation began.

**The user message — the actual work** — The user message is the task: a Python function with three real issues. The parameter name id shadows Python's built-in id() function. The function calls load_all_users() on every invocation — an O(n) full scan when a direct lookup would do. And it returns None implicitly when no match is found, which will crash any caller expecting a dict. The model already knows its role from the system prompt — the user message is simply the material to act on.

**The output — persona shapes every word** — No 'Great function!' opener. No encouraging close. Three bullets, each naming a specific issue precisely. The persona eliminated all filler. If you sent this exact same user message with no system prompt, the same model would open with praise, offer suggestions rather than direct statements, and run twice as long. The system prompt produced this disciplined output — not the user message.



In [ ]:
from openai import OpenAI

OPENAI_API_KEY="Your API KEY"


client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a senior Python engineer doing a code review. "
                "Point out issues as concise bullet points. "
                "Maximum 3 issues. No praise. Be direct and specific."
            )
        },
        {
            "role": "user",
            "content": (
                "Review this function:\n\n"
                "def get_user(id):\n"
                "    users = load_all_users()\n"
                "    for u in users:\n"
                "        if u['id'] == id: return u"
            )
        }
    ]
)

print(response.choices[0].message.content)

- **Inefficient Data Loading**: The function loads all users every time it's called, which is inefficient. Consider caching or loading users once if they are not frequently changing.
- **Missing Return for Non-existent User**: If no user with the specified `id` is found, the function implicitly returns `None`, which could be unclear. It's better to explicitly return `None` for clarity.
- **Lack of Input Validation**: The function does not validate the type of `id`. Ensure it’s of the expected type (e.g., integer or string) to avoid potential runtime errors.


### ✏️ Try It Yourself

Modify the code above — some ideas:
- Change the input values and observe the output
- Add error handling
- Extend the example with one additional feature


In [ ]:
# Your turn — experiment here (Example 1)


## What Makes an Effective Persona

A persona is a system prompt that defines who the AI is, not just what it should do. It includes a role, a communication style, output format rules, and hard constraints on scope.

### Key Points

- role, format, constraints, and focus
- vague personas produce inconsistent results
- specific enough a new employee could follow
- personas are reusable across thousands of messages

> 💡 **Analogy:** Hiring a contractor without a spec means you get whatever they decide. Specifying 'install recessed lighting in the kitchen, four-inch cans, spaced 24 inches apart, and clean up before you leave' means you get exactly what you needed. A persona is the spec sheet you write before the contractor starts. The more precise the spec, the more predictable the result.


## Structured JSON Output

Instruct the model to extract structured data and return clean JSON — ready for your application to use directly, no parsing required

### Step-by-Step Breakdown

**response_format — enforcing JSON at the API level** — The response_format parameter tells the OpenAI API to guarantee that the model's output is parseable JSON. Without this, even if the system prompt says respond with JSON only, the model might occasionally wrap the JSON in a sentence or add a markdown code fence. Setting response_format to json_object enforces this at the infrastructure level — not just at the prompt level. You get double enforcement: the API contract plus the system prompt.

**System prompt — defining the output schema** — The system prompt names the role — contact data extractor — and lists the exact fields to return: name, title, company, and email. This is your schema definition in plain English. The model treats this field list as the keys it must include in every response. Combined with response_format, you have now specified both the structure of the JSON and its required contents — without writing a single line of schema validation code.

**Parsing the response — structured data in two lines** — json.loads converts the guaranteed-valid response string into a Python dictionary. From here, data is a regular dict — you call data['email'], pass it to a database insert, or send it downstream to another API. No regex, no line-by-line text parsing, no fragile string slicing. The AI handled the natural language understanding. Your application code receives clean, typed, structured data every time.



In [2]:
from openai import OpenAI
import json

client = OpenAI(api_key=OPENAI_API_KEY)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": (
                "You are a contact data extractor. "
                "Always respond with valid JSON only. "
                "Extract these fields: name, title, company, email."
            )
        },
        {
            "role": "user",
            "content": (
                "Process this: John Smith, CTO at Acme Corp — "
                "reach him at john.smith@acme.com"
            )
        }
    ]
)

data = json.loads(response.choices[0].message.content)
print(json.dumps(data, indent=2))

{
  "name": "John Smith",
  "title": "CTO",
  "company": "Acme Corp",
  "email": "john.smith@acme.com"
}
